In [1]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import sys
from pathlib import Path
import json
from scipy.stats import linregress
import data
import run
import models

In [ ]:
ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]
# Walk-forward validation: each of these months is predicted by models trained only
# on the sales before that month, the same way the real test works.
DISCOVERY_MONTHS = pd.period_range(
    "2024-01",
    "2025-11",
    freq="M"
)

extra_columns = [
    "house_style",
    "municipality",
    "community",
    "bedrooms",
    "washrooms",
    "parking_total",
    "garage",
    "lot_front",
    "lot_depth"
]


In [ ]:

listings = data.load("listings")


results = []

for month in DISCOVERY_MONTHS:
    train = listings[
        listings["date_sold"] < month.start_time
    ]

    homes = listings[
        listings["date_sold"].dt.to_period("M") == month
    ]

    if homes.empty:
        continue

    # Only fit the off-market model
    model = models.make_model("off_market").fit(train)

    inputs = data.hide_answers(homes, "off_market")
    predicted = model.predict(inputs)

    # Remove interaction contribution to recover baseline predictions
    X, coordinates, months = model._prepare(inputs, fit=False)

    interaction_log = model.interaction_model.predict(
        X, coordinates, months
    )

    baseline_log = np.log(predicted) - interaction_log

    results.append(pd.DataFrame({
        "ml_num": homes["ml_num"].to_numpy(),
        "month": str(month),
        "house_type_name": homes["house_type_name"].to_numpy(),
        "latitude": homes["latitude"].to_numpy(),
        "longitude": homes["longitude"].to_numpy(),
        "date_listed": homes["date_listed"].to_numpy(),
        "sold_price": homes["sold_price"].to_numpy(),
        "log_residual": (
            np.log(homes["sold_price"].to_numpy())
            - baseline_log
        )
    }))

    print(f"{month}: {len(homes):,} observations")

discovery = pd.concat(results, ignore_index=True)


2024-01: 1,460 observations
2024-02: 1,947 observations
2024-03: 2,250 observations
2024-04: 2,631 observations
2024-05: 2,634 observations


c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2024-06: 2,196 observations
2024-07: 1,886 observations
2024-08: 1,723 observations
2024-09: 1,825 observations


c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2024-10: 2,451 observations
2024-11: 2,207 observations
2024-12: 1,345 observations
2025-01: 1,339 observations
2025-02: 1,509 observations
2025-03: 1,931 observations
2025-04: 2,077 observations
2025-05: 2,266 observations
2025-06: 2,283 observations


c:\Users\deant\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


2025-07: 2,068 observations
2025-08: 1,666 observations
2025-09: 2,035 observations
2025-10: 2,250 observations
2025-11: 1,867 observations


In [21]:
discovery_results = discovery.copy()

discovery_results["signed_error"] = (
    100 * np.expm1(-discovery_results["log_residual"])
)
discovery_results["absolute_error"] = (
    discovery_results["signed_error"].abs()
)

monthly_results = (
    discovery_results.groupby("month")
    .agg(
        Homes=("ml_num", "size"),
        MAPE=("absolute_error", "mean"),
        MdAPE=("absolute_error", "median"),
        Mean_Bias=("signed_error", "mean"),
        Median_Bias=("signed_error", "median")
    )
    .round(3)
)

In [22]:
monthly_results

,Homes,MAPE,MdAPE,Mean_Bias,Median_Bias
month,,,,,
2024-01,1460,13.272,10.386,0.694,-1.159
2024-02,1947,12.896,10.414,-2.486,-3.651
2024-03,2250,12.843,10.929,-2.901,-4.000
2024-04,2631,12.876,10.296,-0.783,-1.564
2024-05,2634,13.019,10.458,0.938,-0.209
2024-06,2196,13.859,11.228,3.545,2.332
2024-07,1886,13.957,11.418,5.180,4.506
2024-08,1723,14.192,10.804,4.616,2.947
2024-09,1825,13.850,10.882,4.866,3.131


In [23]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import features

In [ ]:
# Add property size to the discovery observations
d = discovery.merge(
    listings[["ml_num", "house_area"]],
    on="ml_num",
    how="left",
    validate="many_to_one"
)

# Raw features for candidate interactions
t = (
    pd.PeriodIndex(
        pd.to_datetime(d["date_listed"]), freq="M"
    ).asi8 - pd.Period("2025-01", freq="M").ordinal
) / 12

lat = d["latitude"] - d["latitude"].mean()
lon = d["longitude"] - d["longitude"].mean()

area = np.log(features.square_feet(d["house_area"]).clip(lower=1))
area = area.fillna(area.median())

types = pd.get_dummies(
    d["house_type_name"],
    prefix="type",
    drop_first=True,
    dtype=float
)

# Candidate interaction families
candidates = {
    "Property type × time": types.mul(t, axis=0),

    "Area × time": pd.DataFrame({
        "area_time": area * t
    }),

    "Property type × location": pd.concat([
        types.mul(lat, axis=0).add_suffix("_latitude"),
        types.mul(lon, axis=0).add_suffix("_longitude")
    ], axis=1),

    "Area × location": pd.DataFrame({
        "area_latitude": area * lat,
        "area_longitude": area * lon
    }),

    "Location × time": pd.DataFrame({
        "latitude_time": lat * t,
        "longitude_time": lon * t
    })
}

# Reconstruct baseline log predictions
residuals = d["log_residual"].to_numpy()
log_actual = np.log(d["sold_price"].to_numpy())
baseline_log = log_actual - residuals

sale_months = pd.PeriodIndex(d["month"], freq="M")
unique_months = sorted(sale_months.unique())

errors = {
    name: [] for name in
    ["No correction", "Bias only", *candidates]
}

folds = 0

print("Discovery observations:", len(discovery))
print("Number of months:", discovery["month"].nunique())
print("Earliest month:", discovery["month"].min())
print("Latest month:", discovery["month"].max())

print("\nObservations per month:")
print(discovery.groupby("month").size())


min_train_months = 9
test_months = 3

if len(unique_months) <= min_train_months:
    raise ValueError(
        f"Only {len(unique_months)} discovery months available. "
        f"Need at least {min_train_months + 1}. "
        "Check how the discovery DataFrame was generated."
    )

# Expanding window: 9 initial months, then 3-month test blocks
for i in range(9, len(unique_months), 3):
    train_mask = np.asarray(sale_months < unique_months[i])
    test_mask = np.asarray(
        sale_months.isin(unique_months[i:i + 3])
    )

    if not train_mask.any() or not test_mask.any():
        continue

    folds += 1

    corrections = {
        "No correction": np.zeros(test_mask.sum()),
        "Bias only": np.full(
            test_mask.sum(), residuals[train_mask].mean()
        )
    }

    for name, X in candidates.items():
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=100.0)
        )

        model.fit(X.loc[train_mask], residuals[train_mask])
        corrections[name] = model.predict(X.loc[test_mask])

    for name, correction in corrections.items():
        signed_log_error = (
            baseline_log[test_mask]
            + correction
            - log_actual[test_mask]
        )
        ape = 100 * np.abs(np.expm1(signed_log_error))
        errors[name].extend(ape.tolist())

# Construct leaderboard
rows = []

for name, ape in errors.items():
    rows.append({
        "Interaction": name,
        "MAPE": np.mean(ape),
        "MdAPE": np.median(ape),
        "Homes": len(ape),
        "Folds": folds
    })

if folds == 0:
    raise RuntimeError(
        "No interaction-discovery folds were evaluated."
    )

print(f"Completed {folds} folds")

for name, ape in errors.items():
    print(f"{name}: {len(ape)} predictions")

interaction_results = pd.DataFrame(rows)

reference = interaction_results.loc[
    interaction_results["Interaction"] == "Bias only"
].iloc[0]

interaction_results["MAPE_Gain"] = (
    reference["MAPE"] - interaction_results["MAPE"]
)
interaction_results["MdAPE_Gain"] = (
    reference["MdAPE"] - interaction_results["MdAPE"]
)

interaction_results = interaction_results.sort_values(
    ["MdAPE_Gain", "MAPE_Gain"],
    ascending=False
).reset_index(drop=True)

display(interaction_results.round(3))

Discovery observations: 45846
Number of months: 23
Earliest month: 2024-01
Latest month: 2025-11

Observations per month:
month
2024-01    1460
2024-02    1947
2024-03    2250
2024-04    2631
2024-05    2634
2024-06    2196
2024-07    1886
2024-08    1723
2024-09    1825
2024-10    2451
2024-11    2207
2024-12    1345
2025-01    1339
2025-02    1509
2025-03    1931
2025-04    2077
2025-05    2266
2025-06    2283
2025-07    2068
2025-08    1666
2025-09    2035
2025-10    2250
2025-11    1867
dtype: int64
Completed 5 folds
No correction: 27294 predictions
Bias only: 27294 predictions
Property type × time: 27294 predictions
Area × time: 27294 predictions
Property type × location: 27294 predictions
Area × location: 27294 predictions
Location × time: 27294 predictions


,Interaction,MAPE,MdAPE,Homes,Folds,MAPE_Gain,MdAPE_Gain
0,No correction,13.787,10.961,27294,5,0.004,0.012
1,Property type × location,13.788,10.967,27294,5,0.004,0.005
2,Bias only,13.792,10.973,27294,5,0.000,0.000
3,Area × location,13.793,10.973,27294,5,-0.001,-0.001
4,Location × time,13.791,10.977,27294,5,0.000,-0.004
5,Area × time,13.576,11.002,27294,5,0.216,-0.029
6,Property type × time,13.568,11.004,27294,5,0.224,-0.032


In [27]:

missing_columns = [col for col in extra_columns if col not in d.columns]

if missing_columns:
    d = d.merge(
        listings[["ml_num", *missing_columns]],
        on="ml_num",
        how="left",
        validate="many_to_one"
    )

In [40]:

# Categorical variables
styles = pd.get_dummies(
    d["house_style"].fillna("Unknown"),
    prefix="style",
    dtype=float
)

municipalities = pd.get_dummies(
    d["municipality"].fillna("Unknown"),
    prefix="municipality",
    dtype=float
)

communities = pd.get_dummies(
    d["community"].fillna("Unknown"),
    prefix="community",
    dtype=float
)

# Numerical variables
numeric = d[
    ["bedrooms", "washrooms", "parking_total", "garage"]
].apply(pd.to_numeric, errors="coerce").fillna(0)

lot_front = d["lot_front"].fillna(0)
lot_depth = d["lot_depth"].fillna(0)
log_lot_area = np.log1p(
    (lot_front * lot_depth).clip(lower=0)
)

# Temporal features
dates = pd.to_datetime(d["date_listed"])
month_of_year = dates.dt.month

sin_time = np.sin(2 * np.pi * month_of_year / 12)
cos_time = np.cos(2 * np.pi * month_of_year / 12)

# t, area, lat, lon and types are from your existing notebook.

def multiply_features(frame, terms):
    """Multiply each column by each supplied numerical term."""
    return pd.concat([
        frame.mul(values, axis=0).add_suffix(f"__{name}")
        for name, values in terms.items()
    ], axis=1)


candidates.update({

    # Nonlinear temporal effects
    "Property type × quadratic time":
        multiply_features(types, {"time_squared": t**2}),

    "Property type × piecewise time":
        multiply_features(types, {
            "time": t,
            "hinge_2024_07": np.maximum(t + 0.5, 0),
            "hinge_2025_01": np.maximum(t, 0),
            "hinge_2025_07": np.maximum(t - 0.5, 0)
        }),

    "Area × quadratic time":
        pd.DataFrame({
            "area_time": area * t,
            "area_time_squared": area * t**2
        }),

    # Seasonality
    "Property type × seasonality":
        multiply_features(types, {
            "season_sin": sin_time,
            "season_cos": cos_time
        }),

    "Municipality × seasonality":
        multiply_features(municipalities, {
            "season_sin": sin_time,
            "season_cos": cos_time
        }),

    # Area-specific market trends
    "Municipality × time":
        multiply_features(municipalities, {"time": t}),

    "Community × time":
        multiply_features(communities, {"time": t}),

    "House style × time":
        multiply_features(styles, {"time": t}),

    # Numerical property effects over time
    "Property attributes × time":
        multiply_features(numeric, {"time": t}),

    "Lot area × time":
        pd.DataFrame({
            "lot_area_time": log_lot_area * t
        }),

    # Nonlinear geographic effects
    "Property type × nonlinear location":
        multiply_features(types, {
            "latitude_squared": lat**2,
            "longitude_squared": lon**2,
            "latitude_longitude": lat * lon
        }),

    "Area × nonlinear location":
        pd.DataFrame({
            "area_lat_squared": area * lat**2,
            "area_lon_squared": area * lon**2,
            "area_lat_lon": area * lat * lon
        }),

    # Three-way interactions
    "Property type × area × time":
        multiply_features(types, {
            "area_time": area * t
        }),

    "Property type × location × time":
        multiply_features(types, {
            "lat_time": lat * t,
            "lon_time": lon * t
        }),

    "Location × quadratic time":
        pd.DataFrame({
            "latitude_time_squared": lat * t**2,
            "longitude_time_squared": lon * t**2
        })
})


In [47]:

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Remove outdated combinations
for name in [
    "All three interaction families",
    "Attributes × time + municipality seasonality + Property attributes × time",
    "Attributes × time + municipality × time + Property attributes × time"
]:
    candidates.pop(name, None)

# Define combined interaction families
candidates.update({
    "Attributes + lot area × time": pd.concat([
        candidates["Property attributes × time"],
        candidates["Lot area × time"]
    ], axis=1),

    "Attributes × time + municipality seasonality": pd.concat([
        candidates["Property attributes × time"],
        candidates["Municipality × seasonality"]
    ], axis=1),

    "Attributes × time + municipality × time": pd.concat([
        candidates["Property attributes × time"],
        candidates["Municipality × time"]
    ], axis=1),

    "Attributes + lot area + municipality seasonality": pd.concat([
        candidates["Property attributes × time"],
        candidates["Lot area × time"],
        candidates["Municipality × seasonality"]
    ], axis=1),

    "Property type + municipality + attributes × time": pd.concat([
        candidates["Property type × time"],
        candidates["Municipality × time"],
        candidates["Property attributes × time"]
    ], axis=1)
})

# Initialize results
errors = {name: [] for name in ["No correction", "Bias only", *candidates]}
folds = 0

# Walk-forward evaluation
for i in range(9, len(unique_months), 3):
    train_mask = np.asarray(sale_months < unique_months[i])
    test_mask = np.asarray(sale_months.isin(unique_months[i:i + 3]))

    if not train_mask.any() or not test_mask.any():
        continue

    folds += 1

    # Chronological inner CV for RidgeCV
    history_months = sale_months[train_mask]
    inner_cv = [
        (
            np.flatnonzero(history_months < unique_months[j]),
            np.flatnonzero(history_months.isin(unique_months[j:j + 3]))
        )
        for j in range(6, i, 3)
    ]

    corrections = {
        "No correction": np.zeros(test_mask.sum()),
        "Bias only": np.full(
            test_mask.sum(),
            residuals[train_mask].mean()
        )
    }

    for name, X in candidates.items():
        model = make_pipeline(
            StandardScaler(),
            RidgeCV(
                alphas=[1, 10, 100, 1000],
                cv=inner_cv,
                scoring="neg_mean_squared_error"
            )
        )

        model.fit(X.iloc[train_mask], residuals[train_mask])
        corrections[name] = model.predict(X.iloc[test_mask])

    for name, correction in corrections.items():
        ape = 100 * np.abs(np.expm1(
            baseline_log[test_mask]
            + correction
            - log_actual[test_mask]
        ))
        errors[name].extend(ape.tolist())

# Results table
interaction_results = pd.DataFrame([
    {
        "Interaction": name,
        "MAPE": np.mean(ape),
        "MdAPE": np.median(ape),
        "Homes": len(ape),
        "Folds": folds
    }
    for name, ape in errors.items()
])

reference = interaction_results.loc[
    interaction_results["Interaction"] == "No correction"
].iloc[0]

interaction_results["MAPE_Gain"] = (
    reference["MAPE"] - interaction_results["MAPE"]
)
interaction_results["MdAPE_Gain"] = (
    reference["MdAPE"] - interaction_results["MdAPE"]
)

interaction_results = interaction_results.sort_values(
    ["MdAPE_Gain", "MAPE_Gain"],
    ascending=False
).reset_index(drop=True)

with pd.option_context("display.max_colwidth", None):
    display(interaction_results.round(3))


,Interaction,MAPE,MdAPE,Homes,Folds,MAPE_Gain,MdAPE_Gain
0,Property attributes × time,13.553,10.883,27294,5,0.234,0.078
1,Attributes + lot area + municipality seasonality,13.548,10.885,27294,5,0.239,0.075
2,Attributes × time + municipality seasonality,13.535,10.886,27294,5,0.252,0.075
3,Attributes + lot area × time,13.557,10.900,27294,5,0.231,0.060
4,Lot area × time,13.618,10.908,27294,5,0.169,0.053
5,No correction,13.787,10.961,27294,5,0.000,0.000
6,Area × nonlinear location,13.790,10.961,27294,5,-0.003,-0.000
7,Municipality × seasonality,13.727,10.966,27294,5,0.061,-0.005
8,Property type × seasonality,13.708,10.968,27294,5,0.080,-0.008
9,Property type × location,13.788,10.969,27294,5,-0.001,-0.009
